# Session 13.1 Practice: Load Digit Images


## 1. Prepare the digit folders

In Colab, upload `digit_images.zip` through the Files sidebar into the working folder.
Unzip the images, then list the class folders `0`–`9`. The unzip command runs quietly
and replaces any previously extracted copies.
Run the following commands in a Colab code cell before continuing to Section 2.


```python
!unzip -q -o digit_images.zip
!ls sample_data/digit_images
```


## 2. Load images and apply transforms

ImageFolder derives labels from folder names. Compose applies Grayscale, Resize and
ToTensor in order, producing `[1, 28, 28]` tensors with pixel values in `[0, 1]`.
Expect 240 images, ten classes and 24 images per class.


In [ ]:
from torchvision import datasets, transforms

## create the image transform below to convert the images to tensors
digit_transform = transforms.Compose([

])

## finish the code below to load the full dataset of digit images using the datasets.ImageFolder
full_dataset = datasets.ImageFolder( )

class_names = full_dataset.classes
print("Images:", len(full_dataset))
print("Classes:", class_names)
print("Class mapping:", full_dataset.class_to_idx)
print("Images per class:", {name: full_dataset.targets.count(i) for i, name in enumerate(class_names)})

## 3. Inspect tensor information

Retrieve one transformed image and inspect its shape, data type, pixel range and class.
The shape `[1, 28, 28]` means one greyscale channel, height 28 and width 28.


In [ ]:
image, label = full_dataset[0]  # Retrieve one transformed image and its label.
print("Tensor shape:", tuple(image.shape))
print("Data type:", image.dtype)
print("Pixel range:", float(image.min()), "to", float(image.max()))
print("Class:", class_names[label])


## 4. Feed the digit images into a trained CNN model

Split the images into 80% training and 20% testing. Each DataLoader supplies its complete subset in one batch.


In [ ]:
import torch
from torch.utils.data import random_split, DataLoader

SEED = 42
torch.manual_seed(SEED)  # Reproduce the random split and initial model weights.
train_size = int(0.8 * len(full_dataset))  # Use 80% of the images for training.
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])  # Create disjoint random subsets.

train_loader = DataLoader(
    train_dataset, batch_size=len(train_dataset), shuffle=True,  # Put all training images in one shuffled batch.
)
test_loader = DataLoader(
    test_dataset, batch_size=len(test_dataset), shuffle=False,  # Put all test images in one fixed-order batch.
)
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

The CNN accepts images with shape `[N, 1, 28, 28]` and returns scores with shape `[N, 10]`.


In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),  # Produce 16 maps of 28 × 28 for each image.
    nn.ReLU(),  # Replace negative responses with zero.
    nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce each map to 14 × 14.
    nn.Flatten(start_dim=1),  # Flatten each image's maps while keeping the image-count dimension.
    nn.Linear(16 * 14 * 14, len(class_names)),  # Return ten scores per image.
)
LEARNING_RATE = 0.001
loss_function = nn.CrossEntropyLoss()  # Compare raw digit scores with integer class labels.
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Set the learning rate.
print(model)


Train for 30 epochs, using the single training batch and updating the weights once per epoch.


In [ ]:
EPOCHS = 30
REPORT_EVERY = 10
for epoch in range(EPOCHS):
    model.train()  # Select training behaviour.
    for train_images, train_labels in train_loader:  # Retrieve all training images and labels in one batch.
        optimiser.zero_grad()  # Clear gradients from the previous epoch.
        logits = model(train_images)  # Feed the complete training batch to the CNN.
        loss = loss_function(logits, train_labels)  # Calculate the mean loss across the training images.
        loss.backward()  # Calculate parameter gradients.
        optimiser.step()  # Update weights once for this batch.
    if (epoch + 1) % REPORT_EVERY == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}: training loss={loss.item():.4f}")


Use the single test batch to make predictions, then display ten images with their true and predicted labels.


In [ ]:
import matplotlib.pyplot as plt

model.eval()  # Select evaluation behaviour.
with torch.no_grad():  # Test without recording gradients or updating weights.
    for test_images, test_labels in test_loader:  # Retrieve all test images and labels in one batch.
        test_logits = model(test_images)  # Feed the complete test batch to the trained CNN.
        predictions = test_logits.argmax(dim=1)  # Select the highest-scoring class for each image.

correct = (predictions == test_labels).sum().item()
print(f"Test accuracy: {correct / len(test_labels):.1%} ({correct}/{len(test_labels)})")
print("Test input:", tuple(test_images.shape), "Test output:", tuple(test_logits.shape))
figure, axes = plt.subplots(2, 5, figsize=(12, 5))  # Display the first ten test results.
for position, axis in enumerate(axes.flat):
    actual = int(test_labels[position].item())
    predicted = int(predictions[position].item())
    axis.imshow(test_images[position].squeeze(0), cmap="gray", vmin=0, vmax=1)  # Display this test image.
    axis.set_title(f"True: {class_names[actual]} | Pred: {class_names[predicted]}")
    axis.axis("off")
plt.tight_layout()
plt.show()
